# DBSCAN -- UCI Air Quality Dataset

This notebook demonstrates density-based spatial clustering (DBSCAN) applied to air quality
sensor readings, where the goal is to identify clusters of similar pollution patterns without
specifying the number of clusters in advance.

- **Core/border/noise classification:** DBSCAN assigns each point to one of three roles based on
  neighbourhood density
- **Hyperparameter sensitivity:** the effect of `eps` and `min_samples` on cluster structure
- **Arbitrary cluster shapes:** DBSCAN recovers non-convex clusters that confound KMeans
- **Noise detection:** outlier readings are automatically flagged as noise (label = -1)
- All clustering is performed with `mlpackage.DBSCAN`; no sklearn clustering is used

## Mathematical Intuition

### Core, Border, and Noise Points

Given radius $\varepsilon > 0$ and minimum neighbourhood size $m$:

- **Core point:** $|\mathcal{N}_\varepsilon(p)| \geq m$ -- has at least $m$ neighbours within distance $\varepsilon$
- **Border point:** $|\mathcal{N}_\varepsilon(p)| < m$ but $p$ lies within $\varepsilon$ of some core point
- **Noise point:** neither core nor border; assigned label $-1$

### Density-Reachability

Point $q$ is **directly density-reachable** from core point $p$ if:

$$\text{dist}(p, q) \leq \varepsilon$$

A cluster is the maximal set of mutually density-connected points. Because clusters are defined
by connected dense regions, DBSCAN recovers arbitrary shapes -- rings, crescents, interleaved
spirals -- that KMeans cannot separate (KMeans assumes convex, roughly spherical clusters).

### Why DBSCAN Outperforms KMeans on Non-Convex Data

KMeans assigns each point to the nearest centroid, which partitions space into Voronoi regions
(always convex). DBSCAN grows clusters from seed points outward through density chains, so the
cluster boundary can follow any shape that maintains local density $\geq m$ within radius
$\varepsilon$.

## Dataset Overview

**Source:** UCI Air Quality Dataset (ID 360) -- hourly readings from a multi-sensor device in an
Italian city | **Rows used:** 2,000 (subsampled for speed) | **Features used:** CO(GT), NOx(GT)

| Feature | Type | Description |
|---|---|---|
| CO(GT) | Continuous | CO concentration (mg/m3), reference analyser |
| NOx(GT) | Continuous | NOx concentration (ppb), reference analyser |

Negative values in the original dataset represent missing sensor readings and are filtered out.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

from ucimlrepo import fetch_ucirepo
from mlpackage import DBSCAN, KMeans, StandardScaler

aq = fetch_ucirepo(id=360)
df = aq.data.features.copy()
df.dropna(inplace=True)

print("Shape after dropna:", df.shape)
print(df[["CO(GT)", "NOx(GT)"]].describe())

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["CO(GT)"].clip(lower=0), bins=40, color="steelblue", edgecolor="white")
axes[0].set_title("CO(GT) Distribution")
axes[0].set_xlabel("CO (mg/m3)")
axes[0].set_ylabel("Count")

axes[1].hist(df["NOx(GT)"].clip(lower=0), bins=40, color="coral", edgecolor="white")
axes[1].set_title("NOx(GT) Distribution")
axes[1].set_xlabel("NOx (ppb)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## Preprocessing

In [ ]:
mask     = (df["CO(GT)"] > 0) & (df["NOx(GT)"] > 0)
df_clean = df[mask].copy()

rng        = np.random.default_rng(42)
sample_idx = rng.choice(len(df_clean), size=min(2000, len(df_clean)), replace=False)
X_2d       = df_clean[["CO(GT)", "NOx(GT)"]].iloc[sample_idx].values

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_2d)

print(f"Samples used: {X_scaled.shape[0]}")

## Baseline DBSCAN (eps=0.5, min_samples=10)

In [ ]:
db     = DBSCAN(eps=0.5, min_samples=10)
db.fit(X_scaled)
labels = db.labels_

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise    = int(np.sum(labels == -1))
print(f"Clusters found: {n_clusters}  |  Noise points: {n_noise}")

cmap          = plt.get_cmap("tab10")
unique_labels = sorted(set(labels))

plt.figure(figsize=(8, 6))
for lbl in unique_labels:
    m = labels == lbl
    if lbl == -1:
        plt.scatter(X_scaled[m, 0], X_scaled[m, 1],
                    color="black", marker="x", s=20, alpha=0.5, label="Noise")
    else:
        plt.scatter(X_scaled[m, 0], X_scaled[m, 1],
                    color=cmap(lbl % 10), s=15, alpha=0.6, label=f"Cluster {lbl}")
plt.title(f"DBSCAN (eps=0.5, min_samples=10) -- {n_clusters} clusters, {n_noise} noise points")
plt.xlabel("CO(GT) (standardised)")
plt.ylabel("NOx(GT) (standardised)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Hyperparameter Sensitivity

In [ ]:
eps_values = [0.2, 0.5, 1.0, 2.0]
fig, axes  = plt.subplots(1, 4, figsize=(18, 4))

for ax, eps_val in zip(axes, eps_values):
    db_e = DBSCAN(eps=eps_val, min_samples=10)
    db_e.fit(X_scaled)
    lbl  = db_e.labels_
    nc   = len(set(lbl)) - (1 if -1 in lbl else 0)
    nn   = int(np.sum(lbl == -1))
    for l in sorted(set(lbl)):
        m = lbl == l
        if l == -1:
            ax.scatter(X_scaled[m, 0], X_scaled[m, 1], color="black", marker="x", s=10, alpha=0.3)
        else:
            ax.scatter(X_scaled[m, 0], X_scaled[m, 1], color=cmap(l % 10), s=10, alpha=0.5)
    ax.set_title(f"eps={eps_val}\n{nc} clusters, {nn} noise", fontsize=9)
    ax.set_xlabel("CO(GT)")
    ax.set_ylabel("NOx(GT)")
plt.suptitle("DBSCAN: eps Sensitivity (min_samples=10)")
plt.tight_layout()
plt.show()

In [ ]:
ms_values = [5, 10, 20, 50]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, ms in zip(axes, ms_values):
    db_m = DBSCAN(eps=0.5, min_samples=ms)
    db_m.fit(X_scaled)
    lbl  = db_m.labels_
    nc   = len(set(lbl)) - (1 if -1 in lbl else 0)
    nn   = int(np.sum(lbl == -1))
    for l in sorted(set(lbl)):
        m = lbl == l
        if l == -1:
            ax.scatter(X_scaled[m, 0], X_scaled[m, 1], color="black", marker="x", s=10, alpha=0.3)
        else:
            ax.scatter(X_scaled[m, 0], X_scaled[m, 1], color=cmap(l % 10), s=10, alpha=0.5)
    ax.set_title(f"min_samples={ms}\n{nc} clusters, {nn} noise", fontsize=9)
    ax.set_xlabel("CO(GT)")
    ax.set_ylabel("NOx(GT)")
plt.suptitle("DBSCAN: min_samples Sensitivity (eps=0.5)")
plt.tight_layout()
plt.show()

In [ ]:
km       = KMeans(n_clusters=5, random_state=42)
km.fit(X_scaled)
km_labels = km.labels_
db_labels = db.labels_

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for l in sorted(set(db_labels)):
    m = db_labels == l
    if l == -1:
        axes[0].scatter(X_scaled[m, 0], X_scaled[m, 1], color="black", marker="x", s=15, alpha=0.4, label="Noise")
    else:
        axes[0].scatter(X_scaled[m, 0], X_scaled[m, 1], color=cmap(l % 10), s=15, alpha=0.6, label=f"C{l}")
axes[0].set_title("DBSCAN (eps=0.5, min_samples=10)")
axes[0].set_xlabel("CO(GT) (standardised)")
axes[0].set_ylabel("NOx(GT) (standardised)")

for l in range(5):
    m = km_labels == l
    axes[1].scatter(X_scaled[m, 0], X_scaled[m, 1], color=cmap(l), s=15, alpha=0.6, label=f"C{l}")
axes[1].scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
                color="black", marker="X", s=150, zorder=6, label="Centroid")
axes[1].set_title("KMeans (k=5)")
axes[1].set_xlabel("CO(GT) (standardised)")
axes[1].set_ylabel("NOx(GT) (standardised)")

plt.suptitle("DBSCAN vs KMeans on Air Quality Data")
plt.tight_layout()
plt.show()

## Interpretation and Conclusions

- **DBSCAN automatically identifies the number of clusters** rather than requiring it as input;
  the eps and min_samples parameters control what density counts as a cluster, not how many
  clusters exist.
- **Noise detection is built in:** points in low-density regions -- likely sensor anomalies or
  extreme pollution events -- are labelled as outliers rather than forced into the nearest cluster,
  which makes DBSCAN more robust for anomaly detection.
- **eps is the most sensitive hyperparameter:** small eps produces many small clusters and high
  noise counts; large eps merges distinct clusters into one. The choice should be informed by a
  k-distance plot (sort distances to the k-th nearest neighbour and look for the elbow).
- **Increasing min_samples raises the density threshold,** so regions that were dense enough to
  form a cluster at min_samples=5 may be reclassified as noise at min_samples=50.
- **KMeans forces convex, equal-volume partitions** on the same data, splitting the large
  central dense region arbitrarily and misclassifying the scattered high-pollution tail as a
  clean cluster -- demonstrating why density-based methods are preferable for real sensor data.